# BFND: Bangla multimodal fake-news detector (app notebook)

Run this notebook from a fresh runtime with **Runtime → Run all**.

Before you start, these must be in your Google Drive `MyDrive`:
- `Bangla-Multimodal-FakeNews.zip` (base project)
- `BFND_evidence_update.zip` (evidence update)
- `BFND_ocr_filter_update.zip` (OCR filter update)
- `BFND-data/` folder with `features/`, `checkpoints/selection/`, `results/selection/`
- (optional) `evidence_relevance_check.csv` inside `BFND-data/data/annotations/`, filled with 0/1

If a cell fails, fix the cause, then do **Runtime → Restart session** and **Run all** again.
The app is launched with `share=False`, so no public link is created.


In [ ]:
# Cell 1: mount Drive and make sure the app is not already loaded in this runtime
import sys
from google.colab import drive

drive.mount("/content/drive")
assert "app.capstone_app" not in sys.modules, (
    "The app was already loaded in this runtime. Use Runtime -> Restart session, then Run all."
)
print("Drive mounted, runtime is fresh")


In [ ]:
# Cell 2: install the packages the app needs (Colab usually has torch and transformers already)
!pip install -q "gradio>=4" sentence-transformers easyocr pytest
print("Packages installed")


In [ ]:
# Cell 3: paths and the three zip files
import os
import zipfile

DRIVE = "/content/drive/MyDrive"
BFND = f"{DRIVE}/BFND-data"
PROJECT = "/content/Bangla-Multimodal-FakeNews"
BASE_ZIP = f"{DRIVE}/Bangla-Multimodal-FakeNews.zip"
EVIDENCE_ZIP = f"{DRIVE}/BFND_evidence_update.zip"
OCR_ZIP = f"{DRIVE}/BFND_ocr_filter_update.zip"

for path in [BASE_ZIP, EVIDENCE_ZIP, OCR_ZIP]:
    assert os.path.exists(path), f"Zip not found in MyDrive: {path}"
for folder in [f"{BFND}/features", f"{BFND}/checkpoints/selection", f"{BFND}/results/selection"]:
    assert os.path.isdir(folder), f"Folder not found in BFND-data: {folder}"
print("All zip files and BFND-data folders found")


In [ ]:
# Cell 4: extract the base project (only if it is not already there), then apply the two updates
with zipfile.ZipFile(BASE_ZIP) as z:
    top_level = {n.split("/")[0] for n in z.namelist() if n}
nested = top_level == {"Bangla-Multimodal-FakeNews"}

if not os.path.isdir(os.path.join(PROJECT, "app")):
    os.makedirs(PROJECT, exist_ok=True)
    with zipfile.ZipFile(BASE_ZIP) as z:
        z.extractall("/content" if nested else PROJECT)
    print("Base project extracted")

for path in [EVIDENCE_ZIP, OCR_ZIP]:
    with zipfile.ZipFile(path) as z:
        z.extractall(PROJECT)
    print("Applied:", os.path.basename(path))


In [ ]:
# Cell 5: check that the final code is in place
def has_text(relative_path, text):
    with open(os.path.join(PROJECT, relative_path), encoding="utf-8") as f:
        return text in f.read()

checks = {
    "evidence module": os.path.exists(os.path.join(PROJECT, "src/evidence_app.py")),
    "evidence gate script": os.path.exists(os.path.join(PROJECT, "scripts/09_evidence_gate.py")),
    "OCR box filter": has_text("src/ocr.py", "def clean_ocr_boxes"),
    "OCR checkbox in app": has_text("app/capstone_app.py", "শুধু মূল লেখা রাখুন"),
    "Evidence tab in app": has_text("app/capstone_app.py", "10. Evidence (Wikipedia)"),
    "image-only model code": has_text("src/serving.py", "def image_only_probs"),
}
for name, ok in checks.items():
    print("OK  " if ok else "MISS", name)
assert all(checks.values()), "Some code is missing. Check the zip files and run Cell 4 again after a restart."


In [ ]:
# Cell 6: tests for the new parts (shows the result; a failing test stops this cell)
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_evidence_app.py", "tests/test_ocr_clean.py"],
    cwd=PROJECT, check=True,
)


In [ ]:
# Cell 7: relevance gate. It runs only when the sheet is fully labelled.
import subprocess
import sys
import pandas as pd

SHEET = f"{BFND}/data/annotations/evidence_relevance_check.csv"
if not os.path.exists(SHEET):
    print("Relevance sheet not found. The Evidence tab will show NOT VALIDATED.")
else:
    sheet = pd.read_csv(SHEET, dtype=str, keep_default_na=False)
    blank = int((sheet["relevant_0_or_1"].str.strip() == "").sum())
    if blank == 0:
        subprocess.run([sys.executable, "scripts/09_evidence_gate.py"], cwd=PROJECT, check=True)
    else:
        print(f"{blank} rows are not labelled yet. Gate not recomputed; the Evidence tab shows the last saved status.")


In [ ]:
# Cell 8: load the models and start the app (keep this cell running while you use the app)
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

from app.capstone_app import load_runtime, build_ui

load_runtime()
demo = build_ui()
demo.launch(share=False, debug=True)
